# Attention：NumPy、PyTorch 与 JAX

目标：对同一组 Q/K/V 计算因果注意力，验证三种实现数值一致，并直接观察未来 token 的概率为 0。

In [ ]:
import numpy as np

from about_llm.from_scratch.attention_numpy import causal_mask, scaled_dot_product_attention

rng = np.random.default_rng(42)
query = rng.normal(size=(1, 4, 8)).astype(np.float32)
key = rng.normal(size=(1, 4, 8)).astype(np.float32)
value = rng.normal(size=(1, 4, 6)).astype(np.float32)
mask = causal_mask(4)
numpy_output, numpy_probabilities = scaled_dot_product_attention(query, key, value, mask=mask)
print('mask:\n', mask.astype(int))
print('probabilities:\n', np.round(numpy_probabilities[0], 3))
assert np.all(numpy_probabilities[0][np.triu_indices(4, k=1)] == 0)


In [ ]:
import torch

q_t, k_t, v_t = map(torch.from_numpy, (query, key, value))
scores_t = q_t @ k_t.transpose(-2, -1) / (q_t.shape[-1] ** 0.5)
scores_t = scores_t.masked_fill(~torch.from_numpy(mask), -torch.inf)
torch_output = torch.softmax(scores_t, dim=-1) @ v_t
np.testing.assert_allclose(torch_output.numpy(), numpy_output, rtol=1e-5, atol=1e-6)
print('PyTorch matches NumPy:', torch_output.shape)


In [ ]:
import jax
import jax.numpy as jnp

scores_j = jnp.asarray(query) @ jnp.swapaxes(jnp.asarray(key), -2, -1) / (query.shape[-1] ** 0.5)
scores_j = jnp.where(jnp.asarray(mask), scores_j, -jnp.inf)
jax_output = jax.nn.softmax(scores_j, axis=-1) @ jnp.asarray(value)
np.testing.assert_allclose(np.asarray(jax_output), numpy_output, rtol=1e-5, atol=1e-6)
print('JAX matches NumPy:', jax_output.shape)


## 结论

框架不同不改变数学定义。调试时先用小张量和透明 NumPy 基线确认 mask、缩放、softmax 轴与 shape，再讨论 kernel 性能。